# FastAPI Foundations: A Beginner-Friendly Guide

Welcome! This notebook is designed to take you from "What is an API?" to building functional, production-ready services with FastAPI. 

### Why FastAPI?
- **Fast**: It's one of the fastest Python frameworks available (on par with NodeJS and Go).
- **Easy**: Designed to be intuitive and easy to use.
- **Automatic Docs**: It generates interactive documentation (Swagger UI) for you automatically.

## 1. The Basics: Path & Query Parameters

Think of an API endpoint like a function that you can call over the internet. 

- **Path Parameters**: These are part of the URL itself. Use them to identify a *specific* resource (e.g., `items/42`).
- **Query Parameters**: These come after the `?`. Use them for filtering, sorting, or optional settings (e.g., `items/42?q=search_term`).

**Beginner Tip**: FastAPI uses Python "type hints" (like `: int`) to automatically validate your data. If you send a string where an integer is expected, FastAPI will return a clear error message.

In [ ]:
pip install fastapi

In [ ]:
from fastapi import FastAPI

app = FastAPI()

# @app.get defines a "GET" endpoint (used for retrieving data)
@app.get("/items/{item_id}")
async def read_item(item_id: int, q: str | None = None):
    # FastAPI automatically converts item_id to an integer
    # q is optional because we gave it a default value of None
    return {"item_id": item_id, "q": q}

## 2. Handling Data: Request & Response Bodies

When you want to *send* data to the server (like creating a new user or item), you usually send it in the "Request Body" as JSON.

We use **Pydantic** models to define exactly what that data should look like.

**Why this matters**: It acts as a contract. If the incoming data doesn't match your model, FastAPI rejects it before it even touches your logic.

In [ ]:
from pydantic import BaseModel

# Define your data structure
class Item(BaseModel):
    name: str
    price: float
    is_offer: bool | None = None  # Optional field

# @app.post is used for creating new data
@app.post("/items/")
async def create_item(item: Item):
    # item is an instance of the Item class
    return {"message": f"Item '{item.name}' created!", "data": item.model_dump()}

## 3. Dependency Injection: The "Helper" System

Dependency Injection sounds scary, but it's just a way to share logic across different endpoints. 

Think of it as a "plugin" or a "pre-processing" step. For example, you might use it to:
- Check if a user is logged in.
- Get a connection to a database.
- Extract common query parameters.

In [ ]:
from typing import Annotated
from fastapi import Depends

# A reusable function
async def common_parameters(q: str | None = None, skip: int = 0, limit: int = 10):
    return {"q": q, "skip": skip, "limit": limit}

@app.get("/items/")
async def read_items(commons: Annotated[dict, Depends(common_parameters)]):
    # The 'commons' variable will contain the result of common_parameters
    return commons

## 4. Background Tasks: "Do it later"

Sometimes you want to do something that takes a long time (like sending an email or processing an image) *after* you've already sent a response to the user.

**Beginner Tip**: This makes your API feel much faster because the user doesn't have to wait for the long task to finish.

In [ ]:
from fastapi import BackgroundTasks

def log_request(message: str):
    # Pretend this takes a long time or writes to a database
    print(f"LOGGING: {message}")

@app.post("/action/")
async def perform_action(background_tasks: BackgroundTasks):
    # Add the task to the background queue
    background_tasks.add_task(log_request, "A user performed an action!")
    return {"message": "Action started. Logging in background..."}

## 5. Streaming Responses: For LLMs and Big Data

In AI applications, LLMs (like GPT-4) take time to generate text. Instead of waiting for the *whole* response, we "stream" it bit-by-bit as it's generated.

**Why this matters**: This is exactly how ChatGPT shows you text appearing one word at a time!

In [ ]:
from fastapi.responses import StreamingResponse
import asyncio

async def fake_ai_generator():
    words = "This is a streaming response from an imaginary AI model...".split()
    for word in words:
        yield word + " "
        await asyncio.sleep(0.3)  # Wait a bit between words

@app.get("/ai-stream")
async def ai_stream():
    return StreamingResponse(fake_ai_generator())

## 6. Error Handling: Being Polite

When something goes wrong, don't let your app crash. Use `HTTPException` to tell the user *what* went wrong and *why*.

In [ ]:
from fastapi import HTTPException

@app.get("/secure-data/{secret_code}")
async def get_secure_data(secret_code: int):
    if secret_code != 1234:
        # 403 means 'Forbidden'
        raise HTTPException(status_code=403, detail="Wrong secret code!")
    return {"data": "You found the secret treasure!"}